# Serve Qwen3.8-27B on Kaggle free tier

Gives your laptop an **OpenAI-compatible endpoint** on a public HTTPS URL.

## Sidebar, once

**Accelerator = GPU T4 x2**, and **Internet = ON**.

## Two modes, set in the config cell

| Mode | When | Cost |
|---|---|---|
| `BUILD_CACHE = True` | **Run this once, ever** | ~25 min. Downloads the model, the pip wheels and cloudflared into one folder you save as a Dataset |
| `BUILD_CACHE = False` | **Every time after** | ~2 to 3 min to a live endpoint. No downloads at all |

After the one-time build, using this is: click **Run All**, wait ~3 minutes, copy the URL.

## Stopping

Click **Stop Session** in the sidebar when you finish. Nothing else stops the quota. See the notes near the end.

Free quota is roughly **30 GPU-hours a week**, sessions capped near 12 hours.

In [ ]:
# ============================================================
#  CONFIG - the only cell you edit
# ============================================================

BUILD_CACHE = False    # True once to build the cache. False every time after.

# After building, attach the dataset via Add Input and put its path here.
# It should contain: model/  wheels/  bin/cloudflared
CACHE = "/kaggle/input/qwen27b-serving-cache"

RUN_MINUTES = 120      # auto-stop the model after this long
MAX_LEN     = 8192     # lower to 4096 if you hit CUDA out-of-memory

MODEL_REPO  = "philbert440/Qwen3.8-27B-W4A16-AWQ"   # ~19.5 GB, fits 2x T4
# fallback: "cyankiwi/Qwen3.8-27B-AWQ-INT4"          # ~21 GB, more widely used

import os
HAVE_CACHE = (not BUILD_CACHE) and os.path.isdir(CACHE)
print("MODE:", "build cache (one-time)" if BUILD_CACHE
      else ("serve from cache" if HAVE_CACHE else "serve, downloading everything"))
if not BUILD_CACHE and not HAVE_CACHE:
    print(f"  cache not found at {CACHE} - this run will be slow.")
    print("  Set BUILD_CACHE = True once to avoid that in future.")

In [ ]:
# ONE-TIME: build the cache. Skipped unless BUILD_CACHE is True.
#
# Afterwards: sidebar -> Output -> Create Dataset, save /kaggle/working/cache,
# name it qwen27b-serving-cache, then set BUILD_CACHE = False.

if BUILD_CACHE:
    import os, subprocess
    OUT = "/kaggle/working/cache"
    os.makedirs(f"{OUT}/wheels", exist_ok=True)
    os.makedirs(f"{OUT}/bin", exist_ok=True)

    print("1/3  pip wheels for vllm ...")
    subprocess.run(["pip", "download", "vllm", "-d", f"{OUT}/wheels", "-q"], check=True)

    print("2/3  cloudflared binary ...")
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", f"{OUT}/bin/cloudflared"], check=True)

    print("3/3  model weights (~19.5 GB, the slow part) ...")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=MODEL_REPO, local_dir=f"{OUT}/model")

    sz = subprocess.run(["du", "-sh", OUT], capture_output=True, text=True).stdout.strip()
    print("\n" + "="*66)
    print("  CACHE BUILT:", sz)
    print("  Now: sidebar -> Output -> Create Dataset -> save /kaggle/working/cache")
    print("  Name it 'qwen27b-serving-cache', then set BUILD_CACHE = False.")
    print("  Then click Stop Session. Do not run the rest of this notebook now.")
    print("="*66)
    raise SystemExit("cache built - stop here")

In [ ]:
# 1. Hardware check, then install from the cache if we have one.

import torch, subprocess, shutil, os, time
t0 = time.time()

n = torch.cuda.device_count()
assert n >= 2, "Set Accelerator to 'GPU T4 x2' in the sidebar, then Run All again."
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB, compute {p.major}.{p.minor}")

if HAVE_CACHE:
    print("\ninstalling vllm from cached wheels (offline)...")
    subprocess.run(["pip", "install", "-q", "--no-index",
                    "--find-links", f"{CACHE}/wheels", "vllm"], check=True)
    shutil.copy(f"{CACHE}/bin/cloudflared", "/usr/local/bin/cloudflared")
    MODEL_PATH = f"{CACHE}/model"
else:
    print("\nno cache - downloading vllm and cloudflared...")
    subprocess.run(["pip", "install", "-q", "vllm"], check=True)
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"], check=True)
    MODEL_PATH = MODEL_REPO

os.chmod("/usr/local/bin/cloudflared", 0o755)
print(f"setup done in {time.time()-t0:.0f}s | serving from: {MODEL_PATH}")

In [ ]:
# 2. Start the model server.
#    T4 is Turing: no bfloat16, no FlashAttention-2, no Marlin kernels.
#    Hence float16, plain awq, enforce-eager.

import subprocess, secrets

PORT, API_KEY, LOG = 8000, secrets.token_urlsafe(24), "/kaggle/working/vllm.log"

server = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_PATH,
    "--served-model-name", "qwen",
    "--api-key", API_KEY,
    "--port", str(PORT),
    "--tensor-parallel-size", "2",
    "--dtype", "float16",
    "--quantization", "awq",
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", "0.92",
    "--enforce-eager",
    "--limit-mm-per-prompt", "image=0,video=0",
], stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

print("vLLM starting, pid", server.pid)
print("~2 min from cache, ~15 min if downloading")

In [ ]:
# 3. Wait, open the tunnel, print what the laptop needs.

import requests, re, time

def die(msg):
    print(msg + "\n--- last 40 log lines ---")
    print("".join(open(LOG).readlines()[-40:]))
    raise SystemExit(msg)

deadline, shown = time.time() + 45*60, 0
while time.time() < deadline:
    if server.poll() is not None:
        die("vLLM exited during startup.")
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/v1/models",
                        headers={"Authorization": f"Bearer {API_KEY}"}, timeout=5).ok:
            print("model loaded"); break
    except requests.RequestException:
        pass
    lines = open(LOG).readlines()
    for ln in lines[shown:]:
        if any(k in ln for k in ("Loading", "error", "Error", "CUDA", "out of memory", "%|")):
            print(ln.rstrip()[:150])
    shown = len(lines)
    time.sleep(15)
else:
    die("timed out waiting for vLLM.")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT)

PUBLIC = None
for _ in range(40):
    time.sleep(3)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com",
                  open("/kaggle/working/tunnel.log").read())
    if m: PUBLIC = m.group(0); break
assert PUBLIC, "tunnel failed; see /kaggle/working/tunnel.log"

ok = requests.post(PUBLIC + "/v1/chat/completions",
                   headers={"Authorization": f"Bearer {API_KEY}"},
                   json={"model": "qwen", "max_tokens": 8, "temperature": 0,
                         "messages": [{"role": "user", "content": "say: ready"}]},
                   timeout=180).ok

print("\n" + "="*66)
print("  ENDPOINT LIVE" if ok else "  TUNNEL UP, TEST CALL FAILED")
print("="*66)
print(f"  OPENAI_BASE_URL = {PUBLIC}/v1")
print(f"  OPENAI_API_KEY  = {API_KEY}")
print("  model name      = qwen")
print("="*66)
print("  Both change every session. Keep them in a .env, never in git.")

In [ ]:
# 4. Hold the endpoint open, then stop the model automatically.
#    Interrupt this cell to stop sooner.

import time, datetime

end = time.time() + RUN_MINUTES * 60
print(f"Serving until {datetime.datetime.fromtimestamp(end):%H:%M:%S}. "
      f"Interrupt to stop sooner.\n")
try:
    while time.time() < end:
        if server.poll() is not None:
            print("vLLM died:"); print("".join(open(LOG).readlines()[-30:])); break
        print(f"{datetime.datetime.now():%H:%M:%S}  alive, "
              f"{int((end-time.time())/60)} min left  |  {PUBLIC}")
        time.sleep(300)
finally:
    for p in (tunnel, server):
        try: p.terminate()
        except Exception: pass
    time.sleep(5)
    print("\n" + "="*66)
    print("  Model stopped. GPU MEMORY is free.")
    print("")
    print("  YOUR QUOTA IS STILL RUNNING.")
    print("  Kaggle charges for an ATTACHED GPU session, not for GPU use.")
    print("  Click STOP SESSION in the sidebar. That is the only thing")
    print("  that stops the meter. The last cell is a fallback.")
    print("="*66)

## What actually stops the quota

| Action | Frees GPU memory | **Stops quota** |
|---|---|---|
| Timer expires, or you interrupt the cell | Yes | **No** |
| Closing the browser tab | No | **No** |
| **Stop Session** in the sidebar | Yes | **Yes** |
| Kaggle's own idle timeout | Yes | Yes, eventually |

Kaggle bills for a GPU session being **attached**, not for the GPU doing work. The auto-stop protects you from a runaway model. It does **not** end the session.

**Rule: when you finish, click Stop Session. Everything else is a safety net.**

---

## Using it from the laptop

```python
from openai import OpenAI
client = OpenAI(base_url="<OPENAI_BASE_URL>", api_key="<OPENAI_API_KEY>")
print(client.chat.completions.create(
    model="qwen", messages=[{"role": "user", "content": "hello"}]
).choices[0].message.content)
```

## If it fails

| Symptom | Fix |
|---|---|
| CUDA out of memory | `MAX_LEN = 4096`. Still failing: try the cyankiwi model, or a smaller model on one T4 |
| Only 1 GPU visible | Accelerator is not **T4 x2** |
| `pip install --no-index` fails | Kaggle changed its base image, so the cached wheels no longer match. Rebuild the cache |
| Download stuck at 0% | **Internet** is off in the sidebar |
| Marlin / FlashAttention error | Expected on Turing; already avoided by `awq` and `enforce-eager` |
| URL stopped working | Session ended. Run All again; URL and key are new |

## Honest caveats

- **Not yet run on real hardware.** Settings follow from documented T4 limits and the measured 19.5 GB weight size. The first run is the test.
- **~5.5 GB per GPU left for KV cache.** Tight. Fallback is a smaller model on one T4, which is fine here: the LLM only handles language, never computes a number, never decides an action.
- **Cached wheels are tied to Kaggle's Python and CUDA versions.** When Kaggle updates its image the offline install can break, and the cache needs rebuilding. That is the cost of the speed-up.

In [ ]:
# OPTIONAL FALLBACK: end the container from inside the notebook.
#
# Kills every process, terminating the session and stopping the quota.
# Blunt: anything unsaved in /kaggle/working is lost and the notebook will
# look like it crashed. Expected, not a failure.
#
# The Stop Session button is better. Use this only if you cannot reach it.

CONFIRM = False   # set True, then run this cell

if CONFIRM:
    import os, signal
    print("terminating container...")
    os.kill(1, signal.SIGKILL)
else:
    print("Not armed. Set CONFIRM = True to terminate.")
    print("Prefer Stop Session in the sidebar.")